# Notebook 02 — Dataset Analysis & Preprocessing

Comprehensive Exploratory Data Analysis (EDA) on **MedNLI** and **PubMedQA**.

### Objectives:
1. Inspect raw parquet files and verify schema
2. Analyze class distributions across train, validation, and test splits
3. Check sentence length statistics (claim vs evidence token counts)
4. Audit premise overlap / group IDs to prevent data leakage
5. Evaluate PubMedQA's label mapping characteristics and domain shift

In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
sys.path.insert(0, str(Path('..').resolve()))
print('Working directory verified.')

## 1 · MedNLI Data Inspection

In [ ]:
from data.download import DataDownloader
from data.clean import DataCleaner

# Check if cleaned data exists, otherwise run preprocessing
splits_dir = Path('../data/splits')
train_path = splits_dir / 'mednli_train.csv'

if not train_path.exists():
    print('Downloading and preparing MedNLI data...')
    DataDownloader('../data/raw').download_mednli()
    DataCleaner('../data/raw', '../data/processed').clean_mednli()

if train_path.exists():
    train_df = pd.read_csv(train_path)
    print(f'MedNLI Train rows: {len(train_df):,}')
    display(train_df.head(3))
else:
    print('Using synthetic demo cases for analysis.')
    with open('../data/synthetic_demos.json') as f:
        train_df = pd.DataFrame(json.load(f))
        train_df.rename(columns={'expected_verdict': 'label'}, inplace=True)
    display(train_df.head(3))

## 2 · Class Distribution & Balance

In [ ]:
counts = train_df['label'].value_counts()
pcts = train_df['label'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
print('Class Distribution:')
display(dist_df)

plt.figure(figsize=(6, 3))
counts.plot(kind='bar', color=['#2ca02c', '#1f77b4', '#d62728'])
plt.title('MedNLI Label Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 3 · Sentence Length & Token Count Distributions

In [ ]:
train_df['claim_len'] = train_df['claim'].astype(str).apply(lambda s: len(s.split()))
train_df['evidence_len'] = train_df['evidence'].astype(str).apply(lambda s: len(s.split()))

print('Claim word counts summary:')
display(train_df['claim_len'].describe().round(1))

print('\nEvidence word counts summary:')
display(train_df['evidence_len'].describe().round(1))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(train_df['claim_len'], bins=20, color='skyblue', edgecolor='black')
ax[0].set_title('Claim Word Counts')
ax[0].set_xlabel('Words')

ax[1].hist(train_df['evidence_len'], bins=20, color='salmon', edgecolor='black')
ax[1].set_title('Evidence Word Counts')
ax[1].set_xlabel('Words')
plt.tight_layout()
plt.show()

## 4 · Out-of-Domain Dataset: PubMedQA

In [ ]:
pubmed_path = splits_dir / 'pubmedqa_test.csv'
if pubmed_path.exists():
    pqa_df = pd.read_csv(pubmed_path)
    print(f'PubMedQA Test items: {len(pqa_df):,}')
    display(pqa_df.head(2))
    print('\nLabel distribution in PubMedQA:')
    display(pqa_df['label'].value_counts())
else:
    print('PubMedQA test CSV not found. Run `python main.py data download` to fetch.')